# Phase 4 — LangGraph Multi-Agent Investigation

**Project:** Fraud AI Investigator — MENA Fintech Portfolio  
**Notebook:** `notebooks/langgraph_investigation.ipynb`  
**Author:** Ahmed Raza  
**Last updated:** 2026-05

---

## Objective

Validate and analyse the LangGraph multi-agent investigation pipeline.

1. **Does the graph compile and run correctly?** — structure validation
2. **Do agents run in parallel?** — timing analysis (kyc + sanctions simultaneously)
3. **How does the synthesis LLM compare to rule-based triage?** — score correlation
4. **What does a complete investigation case look like?** — end-to-end case review

## Architecture

```
START
  │
  ▼
transaction_agent       ← loads tx data, extracts signals
  │
  ├─────────────────────────────┐
  ▼                             ▼
kyc_agent (parallel)   sanctions_agent (parallel)
  │                             │
  └──────────────┬──────────────┘
                 │
          [conditional]
          wallet? → crypto_agent
                 │
                 ▼
          synthesis_agent    ← LLM final assessment
                 │
                END
```

## Prerequisites

```bash
uv pip install langgraph langchain-core
uv run uvicorn app.main:app --reload
uv run jupyter notebook notebooks/langgraph_investigation.ipynb
```

In [ ]:
import sys
import json
import time
import warnings
from datetime import datetime
from pathlib import Path

import requests
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
warnings.filterwarnings('ignore', category=DeprecationWarning)

plt.style.use('seaborn-v0_8-whitegrid')

BASE_URL        = 'http://localhost:8000'
SCREENSHOTS_DIR = PROJECT_ROOT / 'doc' / 'Screenshots'
SCREENSHOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root : {PROJECT_ROOT}')
print(f'Run timestamp: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

---
## Section 1 — API health and graph validation

In [ ]:
try:
    health = requests.get(f'{BASE_URL}/health', timeout=5).json()
except Exception:
    raise RuntimeError('API not running. Start: uv run uvicorn app.main:app --reload')

assert health['status'] == 'ok'
print('API HEALTH')
print('=' * 45)
for k, v in health['llm_providers'].items():
    print(f"  {k:<12}: {'✓ configured' if v else '✗ missing'}")

stats = requests.get(f'{BASE_URL}/v1/investigate/stats').json()
print()
print('INVESTIGATION STATS')
print(f"  Graph compiled : {stats['graph_compiled']}")
print(f"  Total alerts   : {stats['total_alerts']}")
assert stats['graph_compiled'], 'Graph not compiled — check server startup logs'

---
## Section 2 — End-to-end investigation

**Purpose:** Run the complete pipeline and time each stage.
We generate alerts, triage them, then investigate with LangGraph.

In [ ]:
timings = {}

# Step 1: Generate alerts
print('Step 1: Generating alerts...')
t0  = time.time()
gen = requests.post(f'{BASE_URL}/v1/alerts/generate', json={'limit': 10}).json()
timings['generate'] = time.time() - t0
print(f"  Created: {gen['alerts_created']} alerts in {timings['generate']:.2f}s")

# Step 2: Triage alerts (LLM scores each)
print('\nStep 2: Triaging alerts (LLM scoring)...')
t0     = time.time()
triage = requests.post(f'{BASE_URL}/v1/triage/batch', json={'max_alerts': 3}, timeout=120).json()
timings['triage'] = time.time() - t0
print(f"  Triaged: {triage['processed']} alerts in {timings['triage']:.1f}s")
print(f"  Auto-closed: {triage['auto_closed']} | Investigating: {triage['investigating']}")

# Get an INVESTIGATING alert to investigate
alerts     = requests.get(f'{BASE_URL}/v1/alerts?status=INVESTIGATING&limit=1').json()
target_id  = alerts['alerts'][0]['alert_id'] if alerts['alerts'] else None

if not target_id:
    # If all were auto-closed, pick a PENDING one
    pending = requests.get(f'{BASE_URL}/v1/alerts?status=PENDING&limit=1').json()
    target_id = pending['alerts'][0]['alert_id'] if pending['alerts'] else None

print(f'\nTarget alert for investigation: {target_id}')

# Step 3: Full LangGraph investigation
print('\nStep 3: Running LangGraph investigation (5 agents)...')
t0 = time.time()
inv = requests.post(
    f'{BASE_URL}/v1/investigate/{target_id}',
    json={},
    timeout=60,
).json()
timings['investigation'] = time.time() - t0

print(f'  Completed in {timings["investigation"]:.1f}s')
print(f"  Agents completed : {inv.get('agents_completed', [])}")
print(f"  Final score      : {inv.get('final_risk_score')}")
print(f"  Risk band        : {inv.get('final_risk_band')}")
print(f"  Recommendation   : {inv.get('recommendation')}")
print(f"  Alert status     : {inv.get('alert_status')}")

---
## Section 3 — Pipeline timing visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: pipeline stage timing
ax = axes[0]
stages = ['Alert generation', 'LLM triage', 'LangGraph investigation']
times  = [timings.get('generate', 0), timings.get('triage', 0), timings.get('investigation', 0)]
colors = ['#4CAF50', '#FF9800', '#1565C0']
bars   = ax.barh(stages, times, color=colors, alpha=0.85, edgecolor='white')
for bar, t in zip(bars, times):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            f'{t:.1f}s', va='center', fontsize=11)
ax.set_xlabel('Seconds')
ax.set_title('Pipeline stage latency', fontsize=12)

# Right: agent execution order (conceptual)
ax2 = axes[1]
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 6)
ax2.set_title('LangGraph agent execution order', fontsize=12)

# Draw agent boxes
agent_boxes = [
    (3.5, 5.0, 'transaction_agent', '#1565C0'),
    (1.5, 3.5, 'kyc_agent',         '#4CAF50'),
    (5.5, 3.5, 'sanctions_agent',   '#FF9800'),
    (3.5, 2.0, 'synthesis_agent',   '#D32F2F'),
]
for x, y, name, color in agent_boxes:
    ax2.add_patch(plt.Rectangle((x-1.2, y-0.4), 2.4, 0.8,
                                facecolor=color, alpha=0.8, edgecolor='white', linewidth=1.5))
    ax2.text(x, y, name, ha='center', va='center', fontsize=8,
             color='white', fontweight='bold')

# Draw arrows
arrow_props = dict(arrowstyle='->', color='#333', lw=1.5)
ax2.annotate('', xy=(2.0, 3.9), xytext=(3.2, 4.6), arrowprops=arrow_props)
ax2.annotate('', xy=(5.0, 3.9), xytext=(3.8, 4.6), arrowprops=arrow_props)
ax2.annotate('', xy=(3.5, 2.4), xytext=(2.0, 3.1), arrowprops=arrow_props)
ax2.annotate('', xy=(3.5, 2.4), xytext=(5.0, 3.1), arrowprops=arrow_props)

ax2.text(3.5, 3.5, 'parallel\nexecution', ha='center', va='center',
         fontsize=9, color='#666', style='italic')
ax2.set_xticks([])
ax2.set_yticks([])

plt.suptitle('Phase 4 — LangGraph Investigation Pipeline', y=1.02, fontsize=13, fontweight='bold')
plt.tight_layout()
save_path = SCREENSHOTS_DIR / '09_langgraph_pipeline.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Chart saved: {save_path}')

---
## Section 4 — Investigation narrative review

In [ ]:
# Read the full investigation narrative
result = requests.get(f'{BASE_URL}/v1/investigate/{target_id}/result').json()

print('INVESTIGATION RESULT')
print('=' * 65)
print(f"Alert ID    : {result['alert_id']}")
print(f"Status      : {result['status']}")
print(f"Risk Score  : {result['risk_score']}/100")
print(f"Risk Band   : {result['risk_band']}")
print(f"Trigger     : {result['trigger']}")
print()
print('INVESTIGATION NARRATIVE:')
print('-' * 65)
print(result.get('investigation_summary', 'Not available'))
print()
print('AUDIT TRAIL:')
audit = requests.get(f"{BASE_URL}/v1/alerts/{target_id}/audit").json()
for event in audit['events']:
    print(f"  [{event['timestamp'][:19]}] {event['event_type']:<30} actor={event['actor']}")

---
## Section 5 — Completion checklist

In [ ]:
print('PHASE 4 NOTEBOOK — COMPLETION CHECKLIST')
print('=' * 55)

checks = {
    'API health check passed'              : health['status'] == 'ok',
    'LangGraph graph compiled'             : stats['graph_compiled'],
    'Alerts generated'                     : gen.get('alerts_created', 0) > 0,
    'Triage batch completed'               : triage.get('processed', 0) > 0,
    'Investigation ran (5 agents)'         : bool(inv.get('agents_completed')),
    'Final risk score produced'            : inv.get('final_risk_score') is not None,
    'Recommendation produced'              : inv.get('recommendation') is not None,
    'Pipeline timing chart saved'          : (SCREENSHOTS_DIR / '09_langgraph_pipeline.png').exists(),
    'Audit trail has investigation event'  : any(
        e['event_type'] == 'INVESTIGATION_COMPLETE'
        for e in audit.get('events', [])
    ),
}

all_passed = True
for label, passed in checks.items():
    print(f"  {'✓' if passed else '✗'}  {label}")
    if not passed:
        all_passed = False

print()
if all_passed:
    print('All checks passed — Phase 4 complete ✓')
    print('Ready for Phase 5: HITL review + fraud memory.')
else:
    print('Some checks failed — review sections above.')

print(f'\nCompleted: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')